In [2]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path
from etc.parse_ids import XMLParser

# Get the notebook's directory and go up to project root
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
data_folder = project_root / "data" / "resources"
PilotDC9 = data_folder / "PilotStudy_Afekta"

graph_gml = data_folder / "generated" / "modified_graph.gml"
human1_xml = data_folder / "Human-GEM.xml"

In [3]:
G = nx.read_gml(graph_gml)

In [4]:
chebi_match = pd.read_csv(PilotDC9 / "AFEKTA_chebis.csv")

In [5]:
name_match = pd.read_excel(PilotDC9 / "Supplementary_Material_2_Results_Dennisse_Avella.xlsx",sheet_name="keyIDs")

In [6]:
mask = name_match["Idlevel"] <= 2

# Clean empty values or '-' to None
curated = (
    name_match.loc[mask, "CuratedID"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
name_match.loc[mask, "CuratedID"] = curated.where(curated.notna(), None)
# Clean chebi_match
chebi_clean = chebi_match.copy()
chebi_clean["Input name"] = (
    chebi_clean["Input name"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
chebi_clean["CHEBI_ID"] = chebi_clean["CHEBI_ID"].replace({"": None, "-": None})
# Create lookup table from chebi_clean
lookup = (
    chebi_clean.dropna(subset=["Input name"])
    .drop_duplicates(subset=["Input name"], keep="first")
    .set_index("Input name")["CHEBI_ID"]
)
# Map Chebi_IDs to name_match
name_match.loc[mask, "Chebi_ID"] = name_match.loc[mask, "CuratedID"].map(lookup)

In [7]:
name_match

,Class,CuratedID,Idlevel,DatabaseID,Blood,Plasma,Mitra,Capitainer,Whatman,Chebi_ID
0,Amino acids and derivatives,1-methylhistidine,1.0,HMDB0000001,1,1,1,1,1,50599
1,Fatty acids and derivatives,2-Hydroxybutanoic acid,1.0,HMDB0000008 - HMDB0341410,1,1,1,1,1,1148
2,Other compounds,2-Piperidone,1.0,HMDB0011749,1,1,1,1,1,77761
3,Other compounds,3-Carboxy-4-methyl-5-propyl-2-furanpropionic acid,2.0,HMDB0061112,1,1,1,1,1,41254
4,Amino acids and derivatives,3-methylhistidine,2.0,HMDB0000479,1,1,1,1,1,27596
...,...,...,...,...,...,...,...,...,...,...
188,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN
189,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN
190,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN
191,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN


In [8]:
parser = XMLParser(human1_xml)

In [9]:
df = parser.extract_data()

In [10]:
parser._first_of(df.Identifiers[68], {'kegg.compound'})

In [11]:
df.Identifiers[68]

[['bigg.metabolite', 'nrvnccrn'],
 ['vmhmetabolite', 'nrvnccrn'],
 ['metanetx.chemical', 'MNXM8942'],
 ['inchi',
  'InChI=1S',
  'C31H59NO4',
  'c1-5-6-7-8-9-10-11-12-13-14-15-16-17-18-19-20-21-22-23-24-25-26-31(35)36-29(27-30(33)34)28-32(2,3)4',
  'h12-13,29H,5-11,14-28H2,1-4H3',
  't29-',
  'm0',
  's1']]

In [12]:
df_human1 = parser.get_chebi_numbers()

In [13]:
df_human1 = parser.to_identifier_df()

In [14]:
def _pick_chebi_column(df, preferred):
    for col in preferred:
        if col in df.columns:
            return col
    # Fall back to any column containing 'chebi'
    for col in df.columns:
        if "chebi" in str(col).lower():
            return col
    return None

def _normalize_chebi(series):
    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "-": pd.NA, "nan": pd.NA})
    s = s.str.replace(r"^CHEBI:\s*", "", regex=True)
    s = s.str.replace(r"^CHEBI\s*", "", regex=True)
    return s.dropna()

def _normalize_hmdb(series):
    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "-": pd.NA, "nan": pd.NA})
    s = s.str.upper()
    s = s.str.replace(r"^HMDB\s*", "", regex=True)
    return s.dropna()

def _overlap_counts(left, right, normalize):
    left_set = set(normalize(left))
    right_set = set(normalize(right))
    return len(left_set), len(right_set), len(left_set & right_set)

df_human1_col = _pick_chebi_column(df_human1, ["CHEBI_ID", "chebi_id", "ChEBI", "Chebi_ID", "chebi"])
name_match_col = _pick_chebi_column(name_match, ["Chebi_ID", "CHEBI_ID", "chebi_id", "ChEBI", "chebi"])

print("df_human1 chebi column:", df_human1_col)
print("name_match chebi column:", name_match_col)

if df_human1_col and name_match_col:
    u_left, u_right, u_overlap = _overlap_counts(
        df_human1[df_human1_col],
        name_match[name_match_col],
        _normalize_chebi,
    )
    print("unique df_human1 chebi IDs:", u_left)
    print("unique name_match chebi IDs:", u_right)
    print("matching chebi IDs:", u_overlap)
else:
    print("Could not find chebi columns. Available columns:")
    print("df_human1 columns:", list(df_human1.columns))
    print("name_match columns:", list(name_match.columns))

hmdb_human1_col = "hmdb"
hmdb_name_match_col = "DatabaseID"
print("df_human1 hmdb column:", hmdb_human1_col)
print("name_match hmdb column:", hmdb_name_match_col)

if hmdb_human1_col in df_human1.columns and hmdb_name_match_col in name_match.columns:
    u_left, u_right, u_overlap = _overlap_counts(
        df_human1[hmdb_human1_col],
        name_match[hmdb_name_match_col],
        _normalize_hmdb,
    )
    print("unique df_human1 hmdb IDs:", u_left)
    print("unique name_match hmdb IDs:", u_right)
    print("matching hmdb IDs:", u_overlap)
else:
    print("Could not find hmdb columns. Available columns:")
    print("df_human1 columns:", list(df_human1.columns))
    print("name_match columns:", list(name_match.columns))

df_human1 chebi column: chebi
name_match chebi column: Chebi_ID
unique df_human1 chebi IDs: 1424
unique name_match chebi IDs: 69
matching chebi IDs: 36
df_human1 hmdb column: hmdb
name_match hmdb column: DatabaseID
unique df_human1 hmdb IDs: 709
unique name_match hmdb IDs: 79
matching hmdb IDs: 34


In [15]:
df_human1.chebi

HUMAN1_ID
MAM00001c    15389
MAM00001e    15389
MAM00002c    36740
MAM00002e    36740
MAM00003c    78990
             ...  
MAM01316n    60721
MAM00767n    84503
MAM01435m    16755
MAM01729e      NaN
MAM00853c      NaN
Name: chebi, Length: 8456, dtype: object

In [16]:
if df_human1.chebi and name_match.Chebi_ID:
    print("df_human1 chebi column:")

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
df_human1_col

'chebi'

In [ ]:
name_match_col

'Chebi_ID'